# 00 - Parámetros Globales del Proyecto

**Tesis:** Deep Learning for Modeling Democratic Satisfaction and Socioeconomic Inequality in Ecuador (Latinobarómetro, V-Dem, ENEMDU).

Esta celda se corre primero en cualquier notebook de la fase de Modelado. Fija semillas de reproducibilidad, detecta GPU NVIDIA disponible, define el flag `SAMPLING_MODE` (para iterar rápido en una laptop de capacidad media antes de escalar a un servidor con GPU), y resuelve las rutas del proyecto de forma relativa (funciona tanto si el notebook se abre desde `notebooks/` como si el proyecto se mueve de carpeta).

In [1]:
# ============================================================
# Parámetros globales -- correr esta celda antes que cualquier otra
# ============================================================
import os
import random
from pathlib import Path

import numpy as np
import torch

# --- Reproducibilidad ---
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# --- Hardware: detección automática de GPU NVIDIA ---
USE_GPU = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_GPU else "cpu")
print(f"GPU NVIDIA detectada: {USE_GPU}")
if USE_GPU:
    print(f"  Dispositivo: {torch.cuda.get_device_name(0)}")
else:
    print("  Corriendo en CPU. Se recomienda SAMPLING_MODE=True para iterar más rápido.")

# --- Modo de muestreo ---
# Con SAMPLING_MODE=True, los notebooks de modelado deben tomar una
# muestra reducida y ESTRATIFICADA (por año y por clase del target) del
# dataset, en vez de usar dataset_modelado_personas.csv completo. Es
# para poder iterar rápido en una laptop de capacidad media; cambiar a
# False para las corridas finales, idealmente en el servidor con GPU.
SAMPLING_MODE = False  # True para iterar rápido en laptop; False para corridas finales en servidor con GPU
FRACCION_MUESTRA = 0.2  # 20% de las filas si SAMPLING_MODE=True

# --- Ventana temporal del CNN-LSTM ---
# Decidida en el plan de modelado: 3 años hacia atrás por persona,
# con máscara para encuestados de años iniciales sin historia completa.
VENTANA_TEMPORAL_ANIOS = 3

# --- Métrica reina para ajuste de hiperparámetros ---
METRICA_OPTIMIZACION = "pr_auc"  # PR-AUC como criterio principal; F1 de "Satisfecho" como métrica secundaria de reporte

# --- Rutas del proyecto (relativas, funcionan desde notebooks/ o desde la raíz) ---
RAIZ_PROYECTO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUTA_DATA_RAW = RAIZ_PROYECTO / "data" / "raw"
RUTA_DATA_PROCESSED = RAIZ_PROYECTO / "data" / "processed"
RUTA_DATA_MODELS = RAIZ_PROYECTO / "data" / "models"
RUTA_REPORTS = RAIZ_PROYECTO / "reports"
RUTA_FIGURES = RUTA_REPORTS / "figures"
RUTA_TABLAS = RUTA_REPORTS / "tablas"

RUTA_DATASET_PERSONAS = RUTA_DATA_PROCESSED / "dataset_modelado_personas.csv"
RUTA_PANEL_MACRO = RUTA_DATA_PROCESSED / "panel_macro_anual.csv"

for ruta in (RUTA_DATA_MODELS, RUTA_FIGURES, RUTA_TABLAS):
    ruta.mkdir(parents=True, exist_ok=True)

print(f"\nRAIZ_PROYECTO: {RAIZ_PROYECTO}")
print(f"SAMPLING_MODE: {SAMPLING_MODE} (fracción={FRACCION_MUESTRA if SAMPLING_MODE else 1.0})")
print(f"VENTANA_TEMPORAL_ANIOS: {VENTANA_TEMPORAL_ANIOS}")
print(f"RUTA_DATASET_PERSONAS existe: {RUTA_DATASET_PERSONAS.exists()}")
print(f"RUTA_PANEL_MACRO existe: {RUTA_PANEL_MACRO.exists()}")

GPU NVIDIA detectada: False
  Corriendo en CPU. Se recomienda SAMPLING_MODE=True para iterar más rápido.

RAIZ_PROYECTO: c:\TesisSD
SAMPLING_MODE: False (fracción=1.0)
VENTANA_TEMPORAL_ANIOS: 3
RUTA_DATASET_PERSONAS existe: True
RUTA_PANEL_MACRO existe: True
